# Model Training - Core ML Algorithms (First Layer — Linear Models).  
*Splitting the Dataset for Training*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
import numpy as np


#Load and Encode data
df = pd.read_excel("../jpil_farmers_dataset.xlsx", header=1)
df['Training_Encoded'] = df['Training Attended'].map({'Yes': 1, 'No': 0})

#Separate our Features (X) from our Target (y)
X = df[['Farm Size (ha)', 'Fertilizer Used (kg)', 'Training_Encoded']]
y = df['Yield Change (%)']

#Cut the data into Study questions and Exam questions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

#Let's look at the sizes of our piles
print(f"Total rows in our original bag: {len(df)}")
print(f"Rows given to the model to study (X_train): {len(X_train)}")
print(f"Rows hidden for the final exam (X_test): {len(X_test)}")

***First Layer** - Linear Regression Training Model*

In [ ]:
model= LinearRegression()
model.fit(X_train, y_train)
print('The model has completed its training')

# Model Evaluation  

*Confirming that the trained model actually works - calculating the mean absolute error between y_test and y_pred*

In [ ]:
y_pred=model.predict(X_test)
mae=mean_absolute_error(y_test, y_pred)

print(f'mean absolute error (MAE) {mae:.2f}%')

***Second Layer** - Decision Tree* 

In [ ]:
tree_model=DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

tree_y_pred=tree_model.predict(X_test)
tree_mae=mean_absolute_error(y_test, tree_y_pred)

print(f'Decision Tree Mean Absolute Error (MAE): {tree_mae:.2f}%')

***Second Layer** - Random Forest*

In [ ]:
forest_model= RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)
forest_y_pred= forest_model.predict(X_test)
forest_mae=mean_absolute_error(y_test, forest_y_pred)

print(f'Random Forest (MAE): {forest_mae:.2f}%')

## **Feature Importance**  
*To isolate the true driver of agricultural productivity, I developed a Random Forest machine learning pipeline in Python. The model's feature importance metrics confirm that agricultural training is the primary driver of performance, accounting for 78% of predictive power, while purely physical endowments like farm size (5%) and standalone fertilizer quantity (16%) had minimal impact. This proves that input efficiency—knowing how to apply resources through extension training—matters far more than simply having more land or raw inputs.*

In [ ]:
importance_scores=forest_model.feature_importances_
importance_df=pd.DataFrame({
    'Clue_Name': X.columns,
    'Importance_Scores': importance_scores
})

importance_df=importance_df.sort_values('Importance_Scores', ascending=False)
print(importance_df)

## 5 Fold Validation ##

In [ ]:
scores=cross_val_score(forest_model, X, y, cv=5, scoring='neg_mean_absolute_error')
real_mae_scores=-scores

print(f'==5 FOLD VALIDATION RESULTS==')
print(f'Scores for the 5 exams: {real_mae_scores}')
print(f'Average errors across all 5 exams: {real_mae_scores.mean():.2f}%')
print(f'Consistency check Standard deviation: {real_mae_scores.std():.2f}')

## Feature Engineering ##  
***Combined Training and Fertilizer input as a single variable to see the actual effect of training***

In [ ]:
df = pd.read_excel("../jpil_farmers_dataset.xlsx", header=1)
df["Training_Encoded"] = df["Training Attended"].map({'Yes': 1, 'No':0})

df["Fertilizer_x_Training"] = df['Fertilizer Used (kg)']*df['Training_Encoded']
X_engineered = df[['Farm Size (ha)', 'Fertilizer Used (kg)', 'Training_Encoded', 'Fertilizer_x_Training']]
y = df['Yield Change (%)']

X_train, X_test, y_train, y_test=train_test_split(X_engineered, y, test_size=0.20, random_state=42)

forest_model_eng = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model_eng.fit(X_train, y_train)

eng_y_pred = forest_model_eng.predict(X_test)
eng_mae = mean_absolute_error(y_test, eng_y_pred)

print('==New model with Feature Engineering==')
print(f'New Single-Split Test MAE: {eng_mae:.2f}')

#k-fold on the new engineering feature model
scores_eng = cross_val_score(forest_model_eng, X_engineered, y, cv=5, scoring='neg_mean_absolute_error')
real_scores_eng = -scores_eng

print(f'New score validated average MAE: {real_scores_eng.mean():.2f}')


## New Feature Importance after Feature engineering ##  
***By introducing an interaction term ($\text{Fertilizer} \times \text{Training}$), the Random Forest pipeline decomposed the treatment effect. The model reveals that the joint interaction of physical resource endowment and extension knowledge represents the single largest predictive driver (40.5% feature importance), outstripping standalone training (39.0%) and unguided fertilizer usage (16.8%). This strongly indicates that agricultural interventions achieve optimal scale effects not through independent resource distribution, but through combined capacity building that enhances factor efficiency.***

In [ ]:
# Create a new cell to see the updated feature importance rankings
importance_df_eng= pd.DataFrame({
    'feature': X_train.columns,
    'importance': forest_model_eng.feature_importances_
}).sort_values('importance', ascending=False)

print('==Updated Feature Importance==')
print(importance_df_eng)

## Machine Learning Saved Model ##